In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")

SOURCE_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_bronze.customers"
TARGET_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.customers"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.dq_quarantine"

TRACKED_COLUMNS = ["name", "email", "country", "marketing_opt_in"]

In [0]:
import logging

logger = logging.getLogger("lab4.silver_customers")
logger.setLevel(logging.INFO)
logger.propagate = False
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s - %(message)s"))
    logger.addHandler(handler)

In [0]:
bronze_df = spark.table(SOURCE_TABLE)

##### Data quality rules
##### Rows that fail any rule are separated from valid rows

In [0]:
dq_rules = (
    F.col("customer_id").isNotNull()
    & F.col("age").between(0, 120)
    & F.col("email").like("%@%")
    & (F.length("country") == 2)
    & F.col("_rescued_data").isNull()
)

is_valid = F.coalesce(dq_rules, F.lit(False))

valid_df = bronze_df.filter(is_valid)
invalid_df = bronze_df.filter(~is_valid)

##### Quarantine
##### Invalid rows are stored in `dq_quarantine`. MERGE prevents duplicates on re-runs.

In [0]:
quarantine_df = invalid_df.select(
    F.lit(SOURCE_TABLE).alias("source_table"),
    F.to_json(F.struct(*invalid_df.columns)).alias("record"),
    F.current_timestamp().alias("quarantined_at"),
)

quarantine_result = (
    DeltaTable.forName(spark, QUARANTINE_TABLE).alias("t")
    .merge(quarantine_df.alias("s"), "t.source_table = s.source_table AND t.record = s.record")
    .whenNotMatchedInsertAll()
    .execute()
    .first()
)
logger.info("Quarantined %d new invalid rows", quarantine_result["num_inserted_rows"])

In [0]:
latest_first = Window.partitionBy("customer_id").orderBy(
    F.col("ingestion_timestamp").desc(),
    F.col("source_filename").desc(),
)

deduped_df = (
    valid_df
    .withColumn("row_num", F.row_number().over(latest_first))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [0]:
silver_df = deduped_df.select(
    "customer_id",
    "name",
    "email",
    "country",
    "age",
    "signup_date",
    "marketing_opt_in",
    "loyalty_tier",
    F.col("ingestion_timestamp").alias("effective_from"),
    "source_filename",
    "ingestion_timestamp",
    F.current_timestamp().alias("processed_at"),
)

changed_condition = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in TRACKED_COLUMNS)

In [0]:
target = DeltaTable.forName(spark, TARGET_TABLE)
current_df = target.toDF().filter(F.col("is_current"))

changed_df = (
    silver_df.alias("s")
    .join(current_df.alias("t"), F.col("s.customer_id") == F.col("t.customer_id"))
    .filter(F.expr(changed_condition))
    .select("s.*")
)

staged_df = (
    silver_df.withColumn("merge_key", F.col("customer_id"))
    .unionByName(changed_df.withColumn("merge_key", F.lit(None).cast("int")))
)

##### MERGE - SCD Type 2
##### Changed customers: the current version is closed and a new version is inserted. New customers are inserted. Unchanged customers are skipped.

In [0]:
insert_values = {column: f"s.{column}" for column in silver_df.columns}
insert_values["effective_to"] = "NULL"
insert_values["is_current"] = "true"

merge_result = (
    target.alias("t")
    .merge(staged_df.alias("s"), "t.customer_id = s.merge_key AND t.is_current = true")
    .withSchemaEvolution()
    .whenMatchedUpdate(
        condition=changed_condition,
        set={
            "effective_to": "s.effective_from - INTERVAL 1 SECOND",
            "is_current": "false",
            "processed_at": "s.processed_at",
        },
    )
    .whenNotMatchedInsert(values=insert_values)
    .execute()
    .first()
)
logger.info(
    "%s: %d versions closed, %d versions inserted",
    TARGET_TABLE,
    merge_result["num_updated_rows"],
    merge_result["num_inserted_rows"],
)

##### Each closed version ends 1 second before the next version starts, so validity ranges never overlap.

![image_1791445783829.png](./screenshots/03_scd2_validity_ranges.png)